# Bài tập chương 2

## 1. Khởi tạo cơ sở dữ liệu và dữ liệu mẫu
Ta cần tạo hai bảng student và course với cấu trúc và dữ liệu như đã cho trong đề bài. Dưới đây là các câu lệnh SQL để tạo bảng và chèn dữ liệu.

In [111]:
import sqlite3
import pandas as pd
# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# -- Tạo bảng student
cursor.execute('''
CREATE TABLE student (
    student_id INT PRIMARY KEY,
    name VARCHAR(255),
    class VARCHAR(255),
    course_id INT,
    score DECIMAL(3, 1)
);
''')

# -- Chèn dữ liệu vào bảng student
cursor.execute("""
INSERT INTO student (student_id, name, class, course_id, score) VALUES
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
(3, 'Pham Van Nam', 'Toan Tin', NULL, 7.9),
(4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
(5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
(6, 'Dang Thuy Linh', 'Kinh Te', 24, 5.5),
(7, 'Bui Ten Dung', 'Kinh Te', 34, 9.2),
(8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
(9, 'Duong Huu Phuc', 'Kinh Te',NULL , 7.2),
(10, 'Cao Thi Hanh', 'May Tinh',NULL , 7.0);
""")

# -- Tạo bảng course
cursor.execute('''
CREATE TABLE course (
    id INT PRIMARY KEY,
    course_name VARCHAR(255)
);
''')

# -- Chèn dữ liệu vào bảng course
cursor.execute("""
INSERT INTO course (id, course_name) VALUES
(12, 'Giai tich'),
(34, 'Thong ke'),
(26, 'Tin hoc');
""")

conn.commit()



## 2. Kết nối hai bảng

### 2.1.Sử dụng tích Decartes



In [112]:
query = """
SELECT *
FROM student
CROSS JOIN course
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


** Nhận xét:
 + Kết quả trả về 30 dòng (10 sinh viên x 3 môn học). Đây là tích Descartes, kết hợp mỗi dòng của bảng `student` với mọi dòng của bảng `course`.


### 2.2. Sử dụng INNER JOIN

In [113]:
query = """
SELECT s.*, c.course_name
FROM student s
INNER JOIN course c
ON s.course_id = c.id
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Ten Dung,Kinh Te,34,9.2,Thong ke


** Nhận xét:
   + Kết quả  trả về 3 dòng. Chỉ những sinh viên có `course_id` khớp với `id` trong bảng `course` mới xuất hiện. Các sinh viên có `course_id` là NULL hoặc không tồn tại trong bảng `course` đã bị loại bỏ.
   + `INNER JOIN` giúp lọc ra những bản ghi có mối quan hệ chặt chẽ giữa hai bảng


### 2.3. Sử dụng LEFT JOIN

In [114]:
query = """
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,Kinh Te,24.0,5.5,None
6,7,Bui Ten Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


** Nhận xét:
    
  + Kết quả trả về 10 dòng, tương ứng với toàn bộ sinh viên trong bảng `student`. Với những sinh viên không có môn học tương ứng, cột `course_name` hiển thị giá trị `NaN`.
  + `LEFT JOIN`  giữ lại toàn bộ danh sách sinh viên và chỉ bổ sung thông tin môn học nếu có, không làm mất dữ liệu của những sinh viên chưa đăng ký môn học.

### 2.4. Sử dụng RIGHT JOIN

In [115]:
query = """
SELECT s.*, c.course_name
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,7.0,Bui Ten Dung,Kinh Te,34.0,9.2,Thong ke
3,NaN,None,None,NaN,NaN,Tin hoc


** Nhận xét:
  + SQLite không hỗ trợ `RIGHT JOIN` nên ta dùng `LEFT JOIN` với thứ tự bảng ngược lại. Kết quả trả về 4 dòng, bao gồm cả môn "Tin học" không có sinh viên đăng ký.
  + Cách này giúp liệt kê đầy đủ các môn học hiện có và xem xét tình hình đăng ký của từng môn.

### 2.5. Sử dụng FULL OUTER JOIN

In [116]:
query = """
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c
ON s.course_id = c.id

UNION

SELECT s.*, c.course_name
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,course_name
0,NaN,None,None,NaN,NaN,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,None
6,6.0,Dang Thuy Linh,Kinh Te,24.0,5.5,None
7,7.0,Bui Ten Dung,Kinh Te,34.0,9.2,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,None


** Nhận xét:
  + Kết quả trả về 11 dòng, bao gồm tất cả sinh viên từ bảng `student` và tất cả các môn học từ bảng `course`. Đây là phép hợp (UNION) của hai phép `LEFT JOIN`.
  + `FULL OUTER JOIN` cho ta nhìn thấy toàn diện nhất, hiển thị cả những sinh viên không có môn học và những môn học không có sinh viên.

## 3. Cập nhật và phân tích dữ liệu course_id

### 3.1. Cập nhật course_id và loại bỏ dữ liệu không tồn tại

In [117]:
query = """
UPDATE student
SET course_id = 26
WHERE course_id IS NULL
"""
conn.execute(query)
conn.commit()

In [118]:
query = """
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
"""
conn.execute(query)
conn.commit()

### 3.2.a. Tổng số sinh viên, điểm trung bình của từng lớp

In [119]:
query = """
SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class
"""
pd.read_sql_query(query, conn)

,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


** Nhận xét:
  +  Kết quả thống kê theo lớp: Lớp Kinh tế có điểm trung bình cao nhất, tiếp theo là Toán Tin và Máy tính.


### 3.2.b. Tổng số sinh viên, điểm trung bình của từng môn học

In [120]:
query = """
SELECT c.course_name,
       COUNT(s.student_id) AS total_students,
       ROUND(AVG(s.score), 2) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
pd.read_sql_query(query, conn)

,course_name,total_students,avg_score
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37


** Nhận xét:
  + Môn Giải tích có số lượng sinh viên ít nhất (1 sinh viên), điểm trung bình là 6.70 -> mức trung bình khá.
  + Môn Thống kê có 2 sinh viên và đạt điểm trung bình cao nhất (9.20) -> cho thấy kết quả học tập rất tốt.
  + Môn Tin học có 3 sinh viên (nhiều nhất), điểm trung bình 7.37 -> mức khá, phản ánh kết quả ổn định.
  
-> Nhìn chung, điểm trung bình giữa các môn có sự chênh lệch, trong đó Thống kê nổi bật hơn hẳn.

### 3.2.c. Phân loại thi đua theo số điểm của từng môn học b

In [121]:
query = """
SELECT c.course_name,
       ROUND(AVG(s.score), 2) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
           WHEN AVG(s.score) BETWEEN 6.0 AND 8.9 THEN 'Tot'
           ELSE 'Kem'
       END AS classification
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
pd.read_sql_query(query, conn)

,course_name,avg_score,classification
0,Giai tich,6.70,Tot
1,Thong ke,9.20,Xuat sac
2,Tin hoc,7.37,Tot


** Nhận xét:
  + Môn Thống kê được xếp loại Xuất sắc với điểm trung bình 9.20, cho thấy hiệu quả học tập cao.
  + Hai môn Giải tích (6.70) và Tin học (7.37) đều được xếp loại Tốt, thể hiện mức độ đạt và khá ổn định.
  
-> Không có môn nào bị xếp loại Kém, cho thấy chất lượng học tập chung là tích cực.

### 4. Xếp hạng sinh viên:

### 4.a.Theo điểm số

In [122]:
query = """
SELECT student_id, name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Ten Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


** Nhận xét:
  + Sinh viên có điểm cao nhất (9.2) cùng xếp hạng 1. Hàm `RANK()` tự động xử lý các trường hợp bằng điểm (bỏ qua hạng 2, nhảy sang hạng 3).
  
** Kết luận: Xếp hạng chính xác, thể hiện rõ sự chênh lệch và có hiện tượng đồng điểm.

#### Top 3 sinh viên đạt điểm  cao nhất

In [123]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score DESC) AS rnk
    FROM student
)
WHERE rnk <= 3
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,rnk
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Ten Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


** Kết luận:
  + Kết quả đã lọc đúng top 3 theo thứ hạng, nhưng thực tế chỉ có 3 dòng dữ liệu do đồng hạng.
  + Có hiện tượng đồng hạng: 2 sinh viên cùng đạt 9.2 nên đều có rank  1.

#### Top 3 sinh viên đạt điểm thấp nhất

In [124]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score ASC) AS rnk
    FROM student
)
WHERE rnk <= 3
"""
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,rnk
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3


** Nhận xét:
  + Sinh viên Nguyen Minh Hoang có điểm thấp nhất (6.7) -> mức trung bình, cần cải thiện thêm.
  + Sinh viên Cao Thi Hanh đạt 7.0 -> mức trung bình khá, kết quả tương đối ổn định.
  + Sinh viên Duong Huu Phuc đạt 7.2 -> mức khá, cao nhất trong nhóm điểm thấp.

-> Nhóm sinh viên này thuộc top điểm thấp nhất, nhưng mức chênh lệch điểm không lớn, cho thấy mặt bằng chung tương đối đồng đều ở mức trung bình-khá.


### 4.b. Theo điểm số lớp học:

In [125]:
query = """
SELECT student_id, name, class, score,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student
"""
pd.read_sql_query(query, conn)

,student_id,name,class,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Ten Dung,Kinh Te,9.2,1
2,9,Duong Huu Phuc,Kinh Te,7.2,3
3,10,Cao Thi Hanh,May Tinh,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,6.7,2
5,3,Pham Van Nam,Toan Tin,7.9,1


** Nhận xét:
  + Lớp Kinh Tế có 3 sinh viên, trong đó 2 sinh viên đạt điểm cao nhất (9.2) và cùng hạng 1 -> cho thấy kết quả học tập nổi bật; sinh viên còn lại đạt 7.2 xếp hạng 3 do bị nhảy hạng.
  + Lớp Máy Tính có 2 sinh viên, điểm lần lượt 7.0 và 6.7 -> mức trung bình-khá, xếp hạng rõ ràng (1 và 2), không có đồng hạng.
  + Lớp Toán Tin chỉ có 1 sinh viên với điểm 7.9 -> đứng hạng 1 trong lớp, mức khá.

-> Kết quả cho thấy sự chênh lệch giữa các lớp: lớp Kinh Tế có thành tích cao nhất nhưng có hiện tượng đồng hạng, trong khi các lớp còn lại có mức điểm ổn định và phân hóa rõ ràng hơn.

#### Top 3 sinh viên đạt thứ hạng cao nhất theo lớp học

In [126]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, score,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rnk
    FROM student
)
WHERE rnk <= 3
"""
pd.read_sql_query(query, conn)

,student_id,name,class,score,rnk
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Ten Dung,Kinh Te,9.2,1
2,9,Duong Huu Phuc,Kinh Te,7.2,3
3,10,Cao Thi Hanh,May Tinh,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,6.7,2
5,3,Pham Van Nam,Toan Tin,7.9,1


#### Top 3 sinh viên đạt thứ hạng thấp nhất theo lớp

In [127]:
query = """
SELECT *
FROM (
    SELECT student_id, name, class, score,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rnk
    FROM student
)
WHERE rnk <= 3
"""
pd.read_sql_query(query, conn)

,student_id,name,class,score,rnk
0,9,Duong Huu Phuc,Kinh Te,7.2,1
1,2,Tran Thi Lan,Kinh Te,9.2,2
2,7,Bui Ten Dung,Kinh Te,9.2,2
3,1,Nguyen Minh Hoang,May Tinh,6.7,1
4,10,Cao Thi Hanh,May Tinh,7.0,2
5,3,Pham Van Nam,Toan Tin,7.9,1


### 4.c. Điểm số theo mã môn học

In [128]:
query = """
SELECT student_id, name, course_id, score,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
FROM student
"""
pd.read_sql_query(query, conn)

,student_id,name,course_id,score,rank_in_course
0,1,Nguyen Minh Hoang,12,6.7,1
1,3,Pham Van Nam,26,7.9,1
2,9,Duong Huu Phuc,26,7.2,2
3,10,Cao Thi Hanh,26,7.0,3
4,2,Tran Thi Lan,34,9.2,1
5,7,Bui Ten Dung,34,9.2,1


** Nhận xét:
  + Môn (course_id = 12) chỉ có 1 sinh viên với điểm 6.7 → đứng hạng 1, mức trung bình.
  + Môn (course_id = 26) có 3 sinh viên, điểm lần lượt 7.9, 7.2, 7.0 → xếp hạng 1, 2, 3 rõ ràng, cho thấy mức độ phân hóa hợp lý.
  + Môn (course_id = 34) có 2 sinh viên cùng đạt 9.2 và cùng hạng 1 → thể hiện kết quả rất cao nhưng có hiện tượng đồng hạng.

-> Kết quả học tập giữa các môn có sự khác biệt: môn 34 nổi bật với điểm cao nhất nhưng đồng hạng, môn 26 có sự phân hóa rõ ràng, còn môn 12 có ít dữ liệu nên khó đánh giá toàn diện.

#### Top 3 sinh viên đạt thứ hạng cao nhất theo mã môn học

In [129]:
query = """
SELECT *
FROM (
    SELECT s.student_id, s.name, c.course_name, s.score,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS rnk
    FROM student s
    JOIN course c ON s.course_id = c.id
)
WHERE rnk <= 3
"""
pd.read_sql_query(query, conn)

,student_id,name,course_name,score,rnk
0,1,Nguyen Minh Hoang,Giai tich,6.7,1
1,3,Pham Van Nam,Tin hoc,7.9,1
2,9,Duong Huu Phuc,Tin hoc,7.2,2
3,10,Cao Thi Hanh,Tin hoc,7.0,3
4,2,Tran Thi Lan,Thong ke,9.2,1
5,7,Bui Ten Dung,Thong ke,9.2,1


#### Top 3 sinh viên thứ hạng thấp nhất theo mã môn học

In [130]:
query = """
SELECT *
FROM (
    SELECT s.student_id, s.name, c.course_name, s.score,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score ASC) AS rnk
    FROM student s
    JOIN course c ON s.course_id = c.id
)
WHERE rnk <= 3
"""
pd.read_sql_query(query, conn)

,student_id,name,course_name,score,rnk
0,1,Nguyen Minh Hoang,Giai tich,6.7,1
1,10,Cao Thi Hanh,Tin hoc,7.0,1
2,9,Duong Huu Phuc,Tin hoc,7.2,2
3,3,Pham Van Nam,Tin hoc,7.9,3
4,2,Tran Thi Lan,Thong ke,9.2,1
5,7,Bui Ten Dung,Thong ke,9.2,1


### 5. Bổ sung thêm 1 trường graduation_date

#### Thêm cột

In [131]:
query = """
ALTER TABLE student
ADD COLUMN graduation_date DATETIME
"""
conn.execute(query)
conn.commit()

#### Cập nhật dữ liệu

In [132]:
query = """
UPDATE student
SET graduation_date = datetime('now', '+' || rnk || ' days')
FROM (
    SELECT student_id,
           RANK() OVER (ORDER BY score DESC) AS rnk
    FROM student
) AS ranked
WHERE student.student_id = ranked.student_id
"""
conn.execute(query)
conn.commit()

In [133]:
query = "SELECT * FROM student"
pd.read_sql_query(query, conn)

,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-08 03:33:44
1,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-03 03:33:44
2,3,Pham Van Nam,Toan Tin,26,7.9,2026-04-05 03:33:44
3,7,Bui Ten Dung,Kinh Te,34,9.2,2026-04-03 03:33:44
4,9,Duong Huu Phuc,Kinh Te,26,7.2,2026-04-06 03:33:44
5,10,Cao Thi Hanh,May Tinh,26,7.0,2026-04-07 03:33:44


** Nhận xét:
  + Tất cả sinh viên có thời gian tốt nghiệp nằm trong khoảng rất gần nhau (từ 03/04/2026 đến 08/04/2026) -> cho thấy tiến độ học tập khá đồng đều.
  + Không có sự chênh lệch lớn về thời gian tốt nghiệp giữa các sinh viên -> không xuất hiện trường hợp tốt nghiệp sớm hoặc muộn bất thường.
  + Các lớp và môn học khác nhau nhưng thời điểm tốt nghiệp gần như trùng nhau -> có thể cùng một kỳ/đợt tốt nghiệp.

